<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/TrainingModel01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building a simple model using the concepts of Tokenization, Positional Embeddings (Sin/Cos waves), Transformers (Self-Attention) and Feed forward network (ReLU).

# Step 1 — Building a Simple Tokenizer
A tokenizer takes a raw string of text and breaks it into smaller pieces (tokens). For example, Byte Pair Encoding (BPE) which breaks words into subwords — like "unhappiness" → "un", "happi", "ness".

In [1]:
# Step 1: A simple word-level tokenizer

text = "I love code and I love AI"

# Split into words (simplest possible tokenization)
tokens = text.lower().split()
print("Tokens:", tokens)

Tokens: ['i', 'love', 'code', 'and', 'i', 'love', 'ai']


# Step 2 — Building a Vocabulary
A neural network can't read the word "code". It only understands numbers. So we need a lookup table that assigns every unique word an integer ID.

In [2]:
# Step 2: Build a vocabulary (unique words → integer IDs)

# Get unique words, sorted for consistency
vocab = sorted(set(tokens))
print("Vocabulary:", vocab)

# Create two dictionaries — one for each direction
word_to_id = {word: idx for idx, word in enumerate(vocab)}
id_to_word = {idx: word for word, idx in word_to_id.items()}

print("\nWord → ID mapping:")
for word, idx in word_to_id.items():
    print(f"  '{word}' → {idx}")

Vocabulary: ['ai', 'and', 'code', 'i', 'love']

Word → ID mapping:
  'ai' → 0
  'and' → 1
  'code' → 2
  'i' → 3
  'love' → 4


In [3]:
# Convert tokens to IDs
token_ids = [word_to_id[token] for token in tokens]
print("Token IDs:", token_ids)

# Convert to a PyTorch tensor
import torch
token_ids_tensor = torch.tensor(token_ids)
print("As tensor:", token_ids_tensor)

Token IDs: [3, 4, 2, 1, 3, 4, 0]
As tensor: tensor([3, 4, 2, 1, 3, 4, 0])


# Step 3 — The Embedding Layer
Here's where numbers transform into meaning. Right now we have [3, 4, 2, 1, 3, 4, 0] — just flat integers. These don't capture any relationship between words. The number 3 isn't "closer" to 4 than to 0 in any meaningful way.

The embedding layer is essentially a big table with one row per word in our vocabulary, where each row is a vector of numbers

### What nn.Embedding(vocab_size, d_model) does:
It creates a matrix of random numbers with shape (vocab_size, d_model). For GPT with 50,000 tokens and 768 dimensions, it would create a (50,000, 768) table. That's it. Nothing fancy — just a big table of random numbers, one row per word.

```code
Think of it as a massive spreadsheet:
Row 0    (token "the"):  [0.23, 0.87, 0.11, ... 768 numbers]
Row 1    (token "of"):   [0.56, 0.32, 0.78, ... 768 numbers]
Row 2    (token "and"):  [0.41, 0.65, 0.29, ... 768 numbers]
...
Row 49999 (token "zzz"): [0.12, 0.44, 0.93, ... 768 numbers]
```

*Because nn.Embedding is part of nn.Module, PyTorch tracks gradients on that table. So during training, when backpropagation happens, it doesn't update all 50,000 rows. It only updates the rows that were actually looked up in that training step. If your sentence only used words 3, 4, 2, 1, and 0 — only those 5 rows get nudged. The other 49,995 rows stay untouched for that step.*


In [4]:
# Step 3: Embedding layer

vocab_size = len(vocab)     # 5 unique words
d_model = 4                 # each word gets a 4-dimensional vector

# nn.Embedding is just a fancy lookup table
embedding_layer = torch.nn.Embedding(vocab_size, d_model)

print("Embedding table (randomly initialized):")
print(embedding_layer.weight)
print("Shape:", embedding_layer.weight.shape)

Embedding table (randomly initialized):
Parameter containing:
tensor([[ 1.3629, -0.7559, -1.1007,  0.1483],
        [-1.4361, -1.5153,  0.8030,  1.4714],
        [-0.1787, -0.2930,  0.1388, -1.5650],
        [-1.0198, -0.6097, -0.5509,  0.5270],
        [-0.6112,  0.3821,  0.6031,  0.2393]], requires_grad=True)
Shape: torch.Size([5, 4])


### Now what happens when you call embedding_layer(token_ids_tensor)?

This is just a lookup operation. It does NOT create a new table. It goes to the existing table and picks out the rows you asked for.
Exactly what's happening:
```python
# Our token IDs:    [3, 4, 2, 1, 3, 4, 0]
#                    i love code and  i love ai

# What embedding_layer(token_ids_tensor) actually does:
# "Give me row 3" → grabs the vector for 'i'
# "Give me row 4" → grabs the vector for 'love'
# "Give me row 2" → grabs the vector for 'code'
# "Give me row 1" → grabs the vector for 'and'
# "Give me row 3" → grabs the vector for 'i'    (SAME row as before!)
# "Give me row 4" → grabs the vector for 'love' (SAME row as before!)
# "Give me row 0" → grabs the vector for 'ai'
```

The shape should be (7, 4) — 7 tokens in our sentence, each now represented by a 4-dimensional vector. Here's the critical thing: word "i" at position 0 and word "i" at position 4 have the exact same embedding. Because they're the same word, they look up the same row in the table.

And that's a problem. Think about this sentence: "The bank by the river" vs "I went to the bank." The word "bank" should mean different things based on position and context. Embeddings alone can't handle this — they just say "bank is bank."

Context comes from attention. But there's still the position problem — the model needs to know that the first "i" and the second "i" are in different positions.
That's what positional embeddings solve.

In [5]:
# Look up embeddings for our sentence
token_embeddings = embedding_layer(token_ids_tensor)

print("Input IDs:", token_ids_tensor)
print("\nToken embeddings:")
print(token_embeddings)
print("Shape:", token_embeddings.shape)

Input IDs: tensor([3, 4, 2, 1, 3, 4, 0])

Token embeddings:
tensor([[-1.0198, -0.6097, -0.5509,  0.5270],
        [-0.6112,  0.3821,  0.6031,  0.2393],
        [-0.1787, -0.2930,  0.1388, -1.5650],
        [-1.4361, -1.5153,  0.8030,  1.4714],
        [-1.0198, -0.6097, -0.5509,  0.5270],
        [-0.6112,  0.3821,  0.6031,  0.2393],
        [ 1.3629, -0.7559, -1.1007,  0.1483]], grad_fn=<EmbeddingBackward0>)
Shape: torch.Size([7, 4])


# Step 4 — Positional Embeddings
We generate a unique "position signature" for each position using sine and cosine waves of different frequencies. Position 0 gets one pattern, position 1 gets a different pattern, position 6 gets yet another. These patterns are crafted so the model can easily figure out both absolute position ("I'm at position 3") and relative distance ("these two words are 2 positions apart").

In [6]:
# Step 4: Positional Encoding using sin/cos

import math

def create_positional_encoding(max_len, d_model):
    """
    Creates the sin/cos positional encoding table.
    max_len: longest sentence we'll handle
    d_model: embedding dimension (must match token embeddings)
    """
    pe = torch.zeros(max_len, d_model)

    for pos in range(max_len):          # for each position in the sentence
        for i in range(0, d_model, 2):  # for each pair of dimensions
            # The "wavelength" changes with dimension
            denominator = 10000 ** (i / d_model)

            pe[pos, i]     = math.sin(pos / denominator)  # even dimensions
            pe[pos, i + 1] = math.cos(pos / denominator)  # odd dimensions

    return pe

# Create positional encodings for up to 10 positions
pe = create_positional_encoding(max_len=10, d_model=4)

print("Positional encoding for first 7 positions:")
print(pe[:7])
print("Shape:", pe[:7].shape)

Positional encoding for first 7 positions:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992],
        [-0.9589,  0.2837,  0.0500,  0.9988],
        [-0.2794,  0.9602,  0.0600,  0.9982]])
Shape: torch.Size([7, 4])


# Step 5: Combine Token Embeddings + Positional Encodings

### Why adding the positional embeddings to the token embeddings instead of stacking them like (token_embeddings, position,embeddings)?

The deeper reason is about efficiency and information blending. When you add, the position information gets baked into the same 4 numbers that carry meaning. The model is forced to learn embeddings where meaning and position can coexist in the same space. This keeps the model smaller (no extra dimensions) and importantly, when attention later compares two words, it's comparing one vector that contains both "what" and "where" — not two separate pieces it has to figure out how to combine.

In [7]:
# Step 5: Combine token embeddings + positional encodings

# Take only the positions we need (7 tokens)
positional_encoding = pe[:len(tokens)]

# THE KEY OPERATION: add them together
final_embeddings = token_embeddings + positional_encoding

print("Token embeddings (what the word means):")
print(token_embeddings[0], "← first 'i'")
print(token_embeddings[4], "← second 'i'")

print("\nPositional encodings (where the word is):")
print(positional_encoding[0], "← position 0")
print(positional_encoding[4], "← position 4")

print("\nFinal embeddings (meaning + position):")
print(final_embeddings[0], "← 'i' at position 0")
print(final_embeddings[4], "← 'i' at position 4")

print("\nAre the two 'i' embeddings the same?",
      torch.equal(final_embeddings[0], final_embeddings[4]))

print("final_embeddings shape: ", final_embeddings.shape)

Token embeddings (what the word means):
tensor([-1.0198, -0.6097, -0.5509,  0.5270], grad_fn=<SelectBackward0>) ← first 'i'
tensor([-1.0198, -0.6097, -0.5509,  0.5270], grad_fn=<SelectBackward0>) ← second 'i'

Positional encodings (where the word is):
tensor([0., 1., 0., 1.]) ← position 0
tensor([-0.7568, -0.6536,  0.0400,  0.9992]) ← position 4

Final embeddings (meaning + position):
tensor([-1.0198,  0.3903, -0.5509,  1.5270], grad_fn=<SelectBackward0>) ← 'i' at position 0
tensor([-1.7766, -1.2633, -0.5109,  1.5262], grad_fn=<SelectBackward0>) ← 'i' at position 4

Are the two 'i' embeddings the same? False
final_embeddings shape:  torch.Size([7, 4])


# Step-6: Creating Query, Key and Value Matrices

Every word in a sentence has one embedding — its identity. But during
attention, each word needs to play three different roles:

- Query (Q): "What am I looking for?" — what this word wants from others
- Key (K):   "What do I offer?" — what this word advertises about itself  
- Value (V): "What do I actually contain?" — the real information this word shares

We create Q, K, V by multiplying the same embedding by three SEPARATE
weight matrices (W_Q, W_K, W_V). Same word, three different lenses.
These weight matrices start random and get adjusted during training.

In [8]:
torch.manual_seed(42) # To get the same random numbers

# d_model = 4 # embedding size (each word is 4 numbers) - already defined above
d_k = 3       # project into 3 dimensions for Q, K, V

# Weight Matrices
W_Q = torch.randn(d_model, d_k) # shape: (4, 3)
W_K = torch.randn(d_model, d_k) # shape: (4, 3)
W_V = torch.randn(d_model, d_k) # shape: (4, 3)

print("W_Q Shape: ", W_Q.shape)
print("W_K Shape: ", W_K.shape)
print("W_V Shape: ", W_V.shape)

W_Q Shape:  torch.Size([4, 3])
W_K Shape:  torch.Size([4, 3])
W_V Shape:  torch.Size([4, 3])


## What does the Q, K, V vector actually contain?

Each number in a Q, K, or V vector represents some abstract learned
feature — NOT something human-readable like "this number means noun"
or "this number means happiness."

Think of it like coordinates. The embedding [5.6, -1.3, 2.4] is a
POINT in 3D space. The exact numbers don't matter individually. What
matters is WHERE that point sits relative to other points.

Q vector = a point representing "what I'm searching for"
K vector = a point representing "what I'm offering"
V vector = a point representing "what I'll actually share"

Same word, projected to three different locations in space. The model
discovers what these dimensions mean during training — we never define
them. With random weights, the vectors are meaningless. After training,
words that should interact end up with Q and K vectors pointing in
similar directions.

In [9]:
# final_embeddings shape: (7, 4) - 7 words, 4-dim embeddings
# W_Q shape: (4, 3) - projects 4-dim into 3-dim

Q = final_embeddings @ W_Q # (3, 4) @ (4, 3) = (3, 3)
K = final_embeddings @ W_K # (3, 4) @ (4, 3) = (3, 3)
V = final_embeddings @ W_V # (3, 4) @ (4, 3) = (3, 3)

print("Q shape:", Q.shape)  # (7, 3) — 7 tokens, 3-dim queries
print("K shape:", K.shape)
print("V shape:", V.shape)

print("Q (Queries):\n", Q)
print("K (Keys):\n", K)
print("V (Values):\n", V)

Q shape: torch.Size([7, 3])
K shape: torch.Size([7, 3])
V shape: torch.Size([7, 3])
Q (Queries):
 tensor([[-1.0618,  0.5986,  0.6697],
        [ 1.9751, -0.7343,  1.1681],
        [ 0.2821,  0.4867, -0.0808],
        [ 1.4869,  3.4366,  2.5476],
        [-1.6094,  2.3320,  0.8182],
        [ 1.3978, -0.7042,  0.8113],
        [-1.5797,  1.1874,  0.6634]], grad_fn=<MmBackward0>)
K (Keys):
 tensor([[ 0.7116,  2.1523,  1.7962],
        [ 1.5203, -0.0359,  2.3404],
        [-0.3777, -2.1060, -1.5366],
        [-2.7799, -2.5592,  2.3243],
        [-1.7440,  1.2151,  1.2468],
        [-0.7560,  2.6375,  3.9644],
        [ 2.9825, -1.1554, -1.4008]], grad_fn=<MmBackward0>)
V (Values):
 tensor([[ 0.9438, -2.0293, -1.7581],
        [ 0.9104, -0.8448, -0.4139],
        [-0.1150,  0.7663,  0.5321],
        [ 2.3604, -1.8501, -4.0807],
        [ 1.1239, -1.9593, -3.0180],
        [ 0.4634, -0.9027, -1.4837],
        [ 1.2557, -1.8725, -0.4836]], grad_fn=<MmBackward0>)


# Step-7: Computing Attention Scores

Formula:
```code
Attention (Q, K, V) = softmax(QK^T / √d_k) x V
```

### Part-1: QK^T (The Dot Product)

The dot product measures how much two vectors point in the SAME direction.

- Same direction     → large positive number → "highly relevant"
- Perpendicular      → near zero             → "not relevant"  
- Opposite direction → negative number       → "ignore this word"

Q @ K.T compares every word's Query against every word's Key. The
result is a matrix where score[i][j] answers: "how relevant is word j
to what word i is looking for?"

High score = strong match between what word i wants and what word j offers.
Low/negative score = mismatch, word i will mostly ignore word j.

If these scores are wrong (e.g. "I" ignores "AI" in "I love AI"), the
loss function catches the bad prediction and backpropagation adjusts
W_Q and W_K so their Q and K vectors align better next time.

In [10]:
scores = Q @ K.T # (7, 3) @ (3, 7) = (7, 7)

print("Raw attention scores: ")
print(scores)

Raw attention scores: 
tensor([[ 1.7358e+00, -6.8257e-02, -1.8889e+00,  2.9761e+00,  3.4141e+00,
          5.0365e+00, -4.7964e+00],
        [ 1.9232e+00,  5.7630e+00, -9.9440e-01, -8.9609e-01, -2.8804e+00,
          1.2010e+00,  5.1028e+00],
        [ 1.1033e+00,  2.2239e-01, -1.0075e+00, -2.2176e+00, -1.2488e-03,
          7.5032e-01,  3.9214e-01],
        [ 1.3031e+01,  8.0997e+00, -1.1714e+01, -7.0071e+00,  4.7589e+00,
          1.8039e+01, -3.1044e+00],
        [ 5.3437e+00, -6.1545e-01, -5.5608e+00,  4.0743e-01,  6.6604e+00,
          1.0611e+01, -8.6403e+00],
        [ 9.3626e-01,  4.0491e+00, -2.9148e-01, -1.9762e-01, -2.2818e+00,
          3.0231e-01,  3.8459e+00],
        [ 2.6232e+00, -8.9157e-01, -2.9236e+00,  2.8946e+00,  5.0250e+00,
          6.9561e+00, -7.0127e+00]], grad_fn=<MmBackward0>)


### Part-2: Scaling by √d_k

The dot product Q @ K.T gives raw attention scores. But these scores
grow larger as d_k (dimension size) increases — simply because more
numbers are being added together, not because words are more relevant.

Large numbers break softmax — it pushes 99% attention to one word and
ignores everything else. Dividing by √d_k shrinks scores back to a
gentle range so softmax can spread attention across multiple words.

Scaling keeps softmax in its "useful zone" where it can spread attention across multiple words, instead of its "extreme zone" where it picks one word and ignores everything else.

In [11]:
import math
d_k = 3 # dimension of our key vectors
scaled_scores = scores / math.sqrt(d_k)

print("Scaled scores: ")
print(scaled_scores)

Scaled scores: 
tensor([[ 1.0022e+00, -3.9408e-02, -1.0905e+00,  1.7183e+00,  1.9711e+00,
          2.9078e+00, -2.7692e+00],
        [ 1.1104e+00,  3.3273e+00, -5.7411e-01, -5.1736e-01, -1.6630e+00,
          6.9340e-01,  2.9461e+00],
        [ 6.3698e-01,  1.2840e-01, -5.8169e-01, -1.2803e+00, -7.2101e-04,
          4.3320e-01,  2.2640e-01],
        [ 7.5234e+00,  4.6764e+00, -6.7630e+00, -4.0455e+00,  2.7476e+00,
          1.0415e+01, -1.7923e+00],
        [ 3.0852e+00, -3.5533e-01, -3.2105e+00,  2.3523e-01,  3.8454e+00,
          6.1262e+00, -4.9885e+00],
        [ 5.4055e-01,  2.3377e+00, -1.6828e-01, -1.1409e-01, -1.3174e+00,
          1.7454e-01,  2.2204e+00],
        [ 1.5145e+00, -5.1475e-01, -1.6879e+00,  1.6712e+00,  2.9012e+00,
          4.0161e+00, -4.0488e+00]], grad_fn=<DivBackward0>)


### Part-3: Softmax

Raw attention scores are just numbers — some positive, some negative,
no fixed range. Softmax converts them into probabilities between 0 and 1
where each row sums to exactly 1.0.

Now each row is a pie chart: "I give 40% attention here, 35% there,
25% there." This tells us exactly how much each word cares about
every other word.

In [12]:
attention_weights = torch.softmax(scaled_scores, dim=-1)

print("Attention wights:")
print(attention_weights)
print("\nEach row sums to: ", attention_weights.sum(dim=-1))

Attention wights:
tensor([[7.7493e-02, 2.7347e-02, 9.5588e-03, 1.5858e-01, 2.0420e-01, 5.2104e-01,
         1.7839e-03],
        [5.6974e-02, 5.2296e-01, 1.0571e-02, 1.1188e-02, 3.5582e-03, 3.7548e-02,
         3.5720e-01],
        [2.4683e-01, 1.4843e-01, 7.2969e-02, 3.6285e-02, 1.3045e-01, 2.0132e-01,
         1.6371e-01],
        [5.2382e-02, 3.0390e-03, 3.2710e-08, 4.9530e-07, 4.4164e-04, 9.4413e-01,
         4.7145e-06],
        [4.1395e-02, 1.3266e-03, 7.6341e-05, 2.3946e-03, 8.8533e-02, 8.6626e-01,
         1.2901e-05],
        [7.0131e-02, 4.2308e-01, 3.4520e-02, 3.6442e-02, 1.0940e-02, 4.8635e-02,
         3.7625e-01],
        [5.3912e-02, 7.0858e-03, 2.1922e-03, 6.3057e-02, 2.1573e-01, 6.5782e-01,
         2.0681e-04]], grad_fn=<SoftmaxBackward0>)

Each row sums to:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


### Part-4: Multiply by V

Attention weights tell us WHO to listen to. V contains WHAT they're
actually saying. Multiplying weights × V blends the Value vectors
according to those percentages.

If "love" pays 70% attention to "I" and 30% to "code", its output
becomes 70% of I's Value + 30% of code's Value. The word is now
CONTEXTUALIZED — it carries information from the words it found relevant.
This is the entire point of self-attention.

In [13]:
output = attention_weights @ V # (3, 3) @ (3, 3) = (3, 3)

print("Final Output:")
print(output)
print("Shape: ", output.shape)

Final Output:
tensor([[ 0.9444, -1.3402, -2.1798],
        [ 1.0250, -1.2797, -0.5958],
        [ 0.8908, -1.3814, -1.3762],
        [ 0.4902, -0.9620, -1.4955],
        [ 0.5468, -1.0450, -1.6355],
        [ 1.0407, -1.3106, -0.7159],
        [ 0.7535, -1.2473, -1.9810]], grad_fn=<MmBackward0>)
Shape:  torch.Size([7, 3])
